In [ ]:
!pip install torch torchvision pytorch-lightning


In [ ]:
!pip install --upgrade torchmetrics

In [ ]:
import os
import torch
import cv2
import kagglehub
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import pytorch_lightning as pl
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint

In [ ]:
downloaded_data_dir = kagglehub.dataset_download("andrewmvd/car-plate-detection")
print(f"Downloaded to {downloaded_data_dir}")

images_dir = os.path.join(downloaded_data_dir, "images")
annotations_dir = os.path.join(downloaded_data_dir, "annotations")


In [ ]:
class LicensePlateDataset(Dataset):
    def __init__(self, images_dir, annotations_dir):
        self.images_dir = images_dir
        self.annotations_dir = annotations_dir
        self.image_files = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        xml_name = img_name.replace(".png", ".xml").replace(".jpg", ".xml")
        xml_path = os.path.join(self.annotations_dir, xml_name)

        if not os.path.exists(xml_path):
            return self.__getitem__((idx + 1) % len(self))

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            xmin = int(obj.find("bndbox/xmin").text)
            ymin = int(obj.find("bndbox/ymin").text)
            xmax = int(obj.find("bndbox/xmax").text)
            ymax = int(obj.find("bndbox/ymax").text)

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(1)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64)
        }

        img = torch.as_tensor(img, dtype=torch.float32) / 255.0
        img = img.permute(2, 0, 1)

        return img, target

In [ ]:
dataset = LicensePlateDataset(images_dir, annotations_dir)

train_size = int(0.75 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False,
                        collate_fn=collate_fn, num_workers=2, pin_memory=True)

In [ ]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    return interArea / (boxAArea + boxBArea - interArea + 1e-6)

In [ ]:
class FasterRCNNModule(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.model = fasterrcnn_resnet50_fpn(pretrained=True)

        num_classes = 2
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

        self.map_metric = MeanAveragePrecision(iou_type="bbox")

        self.val_outputs = []

    def forward(self, images, targets=None):
        return self.model(images, targets)

    def training_step(self, batch, batch_idx):
        images, targets = batch
        images = [img.to(self.device) for img in images]
        targets = [{k: v.to(self.device) for k, v in t.items()} for t in targets]

        loss_dict = self.model(images, targets)
        loss = sum(loss_dict.values())

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, targets = batch
        images = [img.to(self.device) for img in images]

        outputs = self.model(images)

        preds = []
        gts = []

        for i in range(len(outputs)):
            preds.append({
                "boxes": outputs[i]["boxes"].detach().cpu(),
                "scores": outputs[i]["scores"].detach().cpu(),
                "labels": outputs[i]["labels"].detach().cpu(),
            })

            gts.append({
                "boxes": targets[i]["boxes"].detach().cpu(),
                "labels": targets[i]["labels"].detach().cpu(),
            })

        self.map_metric.update(preds, gts)

        self.val_outputs.append((outputs, targets))

    def on_validation_epoch_end(self):
        metrics = self.map_metric.compute()

        map50 = metrics["map_50"]
        map_all = metrics["map"]

        TP, FP, FN = 0, 0, 0

        for outputs, targets in self.val_outputs:
            for i in range(len(outputs)):
                pred_boxes = outputs[i]["boxes"].detach().cpu().numpy()
                scores = outputs[i]["scores"].detach().cpu().numpy()
                gt_boxes = targets[i]["boxes"].detach().cpu().numpy()

                keep = scores >= 0.5
                pred_boxes = pred_boxes[keep]

                matched = set()

                for pred_box in pred_boxes:
                    found = False
                    for j, gt_box in enumerate(gt_boxes):
                        iou = compute_iou(pred_box, gt_box)
                        if iou >= 0.5 and j not in matched:
                            TP += 1
                            matched.add(j)
                            found = True
                            break
                    if not found:
                        FP += 1

                FN += len(gt_boxes) - len(matched)

        precision = TP / (TP + FP + 1e-6)
        recall = TP / (TP + FN + 1e-6)

        self.log("mAP_50", map50, prog_bar=True)
        self.log("mAP", map_all, prog_bar=False)
        self.log("precision", precision, prog_bar=True)
        self.log("recall", recall, prog_bar=True)

        self.map_metric.reset()
        self.val_outputs.clear()

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=0.005, momentum=0.9)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/logs/faster_rcnn/version_12

In [ ]:
logger = TensorBoardLogger("logs", name="faster_rcnn")

model = FasterRCNNModule()

checkpoint_callback = ModelCheckpoint(
    monitor="mAP",
    mode="max",
    save_top_k=1,
    filename="best-model"
)

trainer = pl.Trainer(
    max_epochs=40,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=logger,
    callbacks=[checkpoint_callback]
)

# trainer.fit(model, train_loader, val_loader)

In [ ]:
trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path="/content/logs/faster_rcnn/version_12/checkpoints/best-model.ckpt"
)